## Inspect a flat RGB PNG dataset


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import torch
from IPython.display import display
from torch.utils.data import Dataset
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

# ── Config ───────────────────────────────────────────
@dataclass
class DatasetConfig:
    dataset_dir: str = "../data/dmlab_topdown_maps"
    num_frames: int = 16
    columns: int = 4

# ── Inputs / Data ────────────────────────────────────
class PNGDataset(Dataset):
    """Load a flat folder of uniform-size RGB PNGs lazily."""

    def __init__(self, folder):
        """Index PNG files without loading their pixels."""
        self.paths = sorted(Path(folder).glob("*.png"))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return read_image(str(self.paths[index]), mode=ImageReadMode.RGB)  # [C, H, W], uint8

# ── Output / IO ──────────────────────────────────────
cfg = DatasetConfig()
dataset = PNGDataset(cfg.dataset_dir)
frame = dataset[0]
print(f"Folder: {Path(cfg.dataset_dir).resolve()}")
print(f"Dataset [N, C, H, W]: {(len(dataset), *frame.shape)} | dtype: {frame.dtype}")
indices = torch.linspace(0, len(dataset) - 1, steps=min(cfg.num_frames, len(dataset))).long().tolist()
frames = torch.stack([dataset[index] for index in indices])  # [B, C, H, W]
print(f"Preview [B, C, H, W]: {tuple(frames.shape)}")
grid = make_grid(frames, nrow=cfg.columns, padding=2)  # [C, grid_H, grid_W]
display(to_pil_image(grid))


## Inspect rollout HDF5 datasets

Datasets whose `.h5` file is missing are skipped.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.ticker import MaxNLocator

# ── Config ───────────────────────────────────────────
@dataclass
class RolloutConfig:
    data_dir: str = "../data"
    num_episodes: int = 4000
    grid_size: int = 128
    seed: int = 101

# name -> (file, state names, action names)
ROLLOUT_SPECS = {
    "DMLab": ("DMLabRolloutDataset.h5", ["x", "y", "yaw"], [f"action_{i}" for i in range(7)]),
}

# ── Inputs / Data ────────────────────────────────────
cfg = RolloutConfig()
available = {name: (Path(cfg.data_dir) / filename, state_names, action_names)
             for name, (filename, state_names, action_names) in ROLLOUT_SPECS.items()}


In [ ]:
# ── Evaluation ───────────────────────────────────────
def print_stats(title, values, names):
    """Print per-channel mean/std/min/max of a [E, T, D] array."""
    print(f"\n[{title}] (flattened across completed episodes & timesteps)")
    print(f"{'Variable':<15} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-" * 63)
    for i, name in enumerate(names):
        x = values[:, :, i].astype(np.float32).ravel()  # float16 rewards overflow in std
        print(f"{name:<15} {x.mean():>11.4f} {x.std():>11.4f} {x.min():>11.4f} {x.max():>11.4f}")


def analyze_hdf5(h5_path, state_names, action_names):
    """Print structure and statistics for one rollout HDF5 dataset."""
    file_size_mb = h5_path.stat().st_size / 1024**2
    print("\n" + "=" * 80 + f"\nDataset: {h5_path.name}\n" + "=" * 80)
    print(f"\n[FILE SIZE] {file_size_mb:.2f} MB")

    with h5py.File(h5_path, "r") as f:
        print("\n[HDF5 STRUCTURE]")
        def print_structure(name, obj):
            if isinstance(obj, h5py.Dataset): print(f"{name}: shape={obj.shape}, dtype={obj.dtype}, size={obj.nbytes / 1024**2:.2f}MB")
            else: print(f"[GROUP] {name}/")
        f.visititems(print_structure)

        print("\n[ATTRIBUTES]")
        for key, val in f.attrs.items(): print(f"  {key}: {val}")

        # ── Episodes ─────────────────────────────────
        episodes = f["episodes"]
        episode_ids = np.flatnonzero(f["written_mask"][...])
        steps_per_ep = episodes["state"].shape[1]
        total_timesteps = len(episode_ids) * steps_per_ep
        print(f"\n[EPISODES] Count: {len(episode_ids)}, Steps/episode: {steps_per_ep}, Total: {total_timesteps}")

        print_stats("STATE STATISTICS", episodes["state"][episode_ids], state_names)  # [E, T, D_state]
        print_stats("ACTION STATISTICS", episodes["action"][episode_ids], action_names)  # [E, T, D_action]

        if "reward" in episodes: print_stats("REWARD STATISTICS", episodes["reward"][episode_ids][..., None], ["reward"])

    print("\n[SUMMARY]")
    print(f"  Total timesteps: {total_timesteps:,}\n  Episodes: {len(episode_ids)}\n  Average trajectory length: {steps_per_ep} steps")
    print(f"  Data storage: {file_size_mb:.2f} MB\n  Data per step: {file_size_mb * 1024**2 / total_timesteps:.0f} bytes")


for name, (path, state_names, action_names) in available.items():
    print(f"\nAnalyzing {name}")
    analyze_hdf5(path, state_names, action_names)


In [ ]:
# ── Visit frequency ──────────────────────────────────
def sample_positions(h5_path, rng):
    """Return world-frame (x, y) of all steps in a random subset of completed episodes, each [E * T]."""
    with h5py.File(h5_path, "r") as f:
        episodes = f["episodes"]
        sampled = np.sort(rng.choice(np.flatnonzero(f["written_mask"][...]), size=cfg.num_episodes, replace=False))
        state = episodes["state"][sampled]  # [E, T, D_state]
        x, y = state[..., 0], state[..., 1]
    return x.ravel(), y.ravel(), len(sampled)


# ── Output / IO ──────────────────────────────────────
PAPER = "#fbfbfb"
INK = "#191b21"
SLATE = "#626b7a"
plt.rcParams.update({
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "text.color": INK,
    "axes.labelcolor": SLATE,
    "xtick.color": SLATE,
    "ytick.color": SLATE,
    "axes.edgecolor": "#dfe3ea",
    "axes.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.monospace": ["DM Mono", "DejaVu Sans Mono"],
    "font.size": 10.5,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
})

rng = np.random.default_rng(cfg.seed)
visit_maps = []
for name, (path, _, _) in available.items():
    x, y, n = sample_positions(path, rng)
    visits, x_edges, y_edges = np.histogram2d(x, y, bins=cfg.grid_size,
                                              range=[[x.min(), x.max()], [y.min(), y.max()]])
    extent = (x_edges[0], x_edges[-1], y_edges[0], y_edges[-1])
    visit_maps.append((name, visits.T / n, extent, n))  # [rows = y, cols = x], mean visits per episode

sample_count = max(n for _, _, _, n in visit_maps)
maximum_visits = max(visits.max() for _, visits, _, _ in visit_maps)
norm = LogNorm(vmin=1 / sample_count, vmax=maximum_visits)
cmap = LinearSegmentedColormap.from_list(
    "swm_density",
    ["#e6e3f6", "#c3bbea", "#8f81d6", "#5b49b8", "#2a2078", "#120d45"],  # light lavender (rare) -> deep indigo (frequent)
    N=256,
)
cmap.set_bad(PAPER)

figure, axes = plt.subplots(1, len(visit_maps), figsize=(6.2 * len(visit_maps), 4.8),
                            constrained_layout=True, squeeze=False, facecolor=PAPER)
images = []
for axis, (name, visits, extent, n) in zip(axes[0], visit_maps):
    image = axis.imshow(np.ma.masked_equal(visits, 0), cmap=cmap, norm=norm,
                        interpolation="nearest", origin="lower", extent=extent, aspect="equal")
    images.append(image)
    axis.set_title(f"{name} · {n:,} episodes", loc="left", pad=9, color=INK)
    axis.set_xlabel("x position")
    axis.set_ylabel("y position")
    axis.xaxis.set_major_locator(MaxNLocator(5))
    axis.yaxis.set_major_locator(MaxNLocator(5))
    axis.tick_params(length=0, pad=5)
    for label in axis.get_xticklabels() + axis.get_yticklabels():
        label.set_fontfamily("monospace")
        label.set_fontsize(8.5)

colorbar = figure.colorbar(images[0], ax=axes[0].tolist(), fraction=0.035, pad=0.03)
colorbar.set_label("Mean visits per episode per bin (log scale)", color=SLATE)
colorbar.ax.tick_params(length=0, labelsize=8.5, colors=SLATE)
for label in colorbar.ax.get_yticklabels():
    label.set_fontfamily("monospace")

plt.show()
plt.close(figure)
